In [109]:
## Clono el repositorio de GitHub en Colab para extraer directamente los archivos
!git clone https://github.com/nataliablancot-cyber/Trabajo-Big-Data-.git

Cloning into 'Trabajo-Big-Data-'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 73 (delta 15), reused 63 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 4.36 MiB | 9.93 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [7]:
##Entro en la carpeta
%cd Trabajo-Big-Data-

/content/Trabajo-Big-Data-/Trabajo-Big-Data-


In [8]:
##Compruebo que están los archivos
!ls -lah data/raw

total 4.7M
drwxr-xr-x 2 root root 4.0K May 12 08:54  .
drwxr-xr-x 3 root root 4.0K May 12 08:54  ..
-rw-r--r-- 1 root root 297K May 12 08:54  API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
-rw-r--r-- 1 root root    1 May 12 08:54  .gitkeep
-rw-r--r-- 1 root root  61K May 12 08:54  Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
-rw-r--r-- 1 root root  892 May 12 08:54  Metadata_Indicator_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
-rw-r--r-- 1 root root  41K May 12 08:54 'nrg_ind_id$defaultview_linear.csv'
-rw-r--r-- 1 root root  23K May 12 08:54  Oil_Bulletin_Duties_and_taxes.xlsx
-rw-r--r-- 1 root root 4.2M May 12 08:54  Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx


Sirve para comprobar si Colab está situado dentro del repositorio correcto y para crear la carpeta data/processed

In [10]:
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta raw:", RAW_DIR.exists())
print("Carpeta processed:", PROCESSED_DIR.exists())

Carpeta raw: True
Carpeta processed: True


In [11]:
## Para poder usar Excel
!pip install openpyxl -q

In [12]:
##Importo las librerías necesarias para el proyecto
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

In [13]:
##pipeline_tracking.json debe guardar cuántos registros entran, cuántos salen y qué limpieza se ha aplicado.
pipeline_tracking = {
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "fases": []
}

def add_tracking(fase, dataset, registros_entrada, registros_salida, eliminados=0, motivo=""):
    pipeline_tracking["fases"].append({
        "fase": fase,
        "dataset": dataset,
        "registros_entrada": int(registros_entrada),
        "registros_salida": int(registros_salida),
        "registros_eliminados": int(eliminados),
        "motivo": motivo
    })

In [14]:
##Como los nombres son largos, lo mejor es buscarlos automáticamente.
files_raw = list(RAW_DIR.glob("*"))

for f in files_raw:
    print(f.name)

Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
.gitkeep
nrg_ind_id$defaultview_linear.csv
Metadata_Indicator_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx
API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
Oil_Bulletin_Duties_and_taxes.xlsx


In [15]:
##Ahora creo variables para cada archivo
gdp_file = list(RAW_DIR.glob("API_NY.GDP.PCAP.CD*.csv"))[0]
energy_file = list(RAW_DIR.glob("nrg_ind_id*.csv"))[0]
tax_file = list(RAW_DIR.glob("Oil_Bulletin_Duties_and_taxes*.xlsx"))[0]
prices_file = list(RAW_DIR.glob("Weekly_Oil_Bulletin_Prices_History*.xlsx"))[0]

print("GDP:", gdp_file)
print("Energía:", energy_file)
print("Impuestos:", tax_file)
print("Precios:", prices_file)

GDP: data/raw/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv
Energía: data/raw/nrg_ind_id$defaultview_linear.csv
Impuestos: data/raw/Oil_Bulletin_Duties_and_taxes.xlsx
Precios: data/raw/Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx


In [16]:
##El archivo API_NY.GDP.PCAP.CD...csv parece venir del World Bank. Normalmente estos archivos tienen unas primeras filas de metadatos, por eso se usa skiprows=4
gdp_raw = pd.read_csv(gdp_file, skiprows=4)

print(gdp_raw.shape)
gdp_raw.head()

(266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356,NaN,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029,NaN,NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448,NaN,NaN


In [17]:
##Ahora lo pasamos a formato largo
entrada = len(gdp_raw)

id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

year_cols = [c for c in gdp_raw.columns if str(c).isdigit()]

gdp_long = gdp_raw.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year",
    value_name="gdp_per_capita"
)

gdp_long = gdp_long.rename(columns={
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Indicator Name": "indicator_name",
    "Indicator Code": "indicator_code"
})

gdp_long["year"] = pd.to_numeric(gdp_long["year"], errors="coerce")
gdp_long["gdp_per_capita"] = pd.to_numeric(gdp_long["gdp_per_capita"], errors="coerce")

antes_limpieza = len(gdp_long)

gdp_long = gdp_long.dropna(subset=["country_code", "year"])

add_tracking(
    fase="extraccion_transformacion",
    dataset="PIB_per_capita",
    registros_entrada=entrada,
    registros_salida=len(gdp_long),
    eliminados=antes_limpieza - len(gdp_long),
    motivo="Conversión de formato ancho a largo y eliminación de filas sin país o año"
)

gdp_long.head()

,country_name,country_code,indicator_name,indicator_code,year,gdp_per_capita
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,186.089204
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,121.936832
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,NaN


In [18]:
##Vamos a inspeccionar nrg_ind_id$defaultview_linear.csv
energy_raw = pd.read_csv(energy_file)

print(energy_raw.shape)
energy_raw.head()

(409, 10)


,DATAFLOW,LAST UPDATE,freq,siec,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2015,12.593,NaN,NaN
1,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2016,20.128,NaN,NaN
2,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2017,38.126,NaN,NaN
3,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2018,20.967,NaN,NaN
4,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2019,31.546,NaN,NaN


In [19]:
energy_raw.columns.tolist()

['DATAFLOW',
 'LAST UPDATE',
 'freq',
 'siec',
 'unit',
 'geo',
 'TIME_PERIOD',
 'OBS_VALUE',
 'OBS_FLAG',
 'CONF_STATUS']

In [20]:
##Ahora hacemos una limpieza generalizada.

entrada = len(energy_raw)

energy = energy_raw.copy()

# Normalizar nombres de columnas
energy.columns = (
    energy.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Quitar filas totalmente vacías
energy = energy.dropna(how="all")

# Quitar duplicados
antes_dup = len(energy)
energy = energy.drop_duplicates()

# Limpiar textos
for col in energy.select_dtypes(include="object").columns:
    energy[col] = energy[col].astype(str).str.strip()
    energy[col] = energy[col].replace({"": np.nan, "nan": np.nan, "None": np.nan})

add_tracking(
    fase="limpieza",
    dataset="dependencia_energetica",
    registros_entrada=entrada,
    registros_salida=len(energy),
    eliminados=entrada - len(energy),
    motivo="Eliminación de filas vacías, duplicados y limpieza de blancos"
)

energy.head()

,dataflow,last_update,freq,siec,unit,geo,time_period,obs_value,obs_flag,conf_status
0,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2015,12.593,NaN,NaN
1,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2016,20.128,NaN,NaN
2,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2017,38.126,NaN,NaN
3,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2018,20.967,NaN,NaN
4,ESTAT:NRG_IND_ID$DEFAULTVIEW(1.0),21/04/26 23:00:00,Annual,Total,Percentage,Albania,2019,31.546,NaN,NaN


In [21]:
##Cambio el nombre de las columnas de geo, time_period y obs_value
energy_clean = energy.rename(columns={
    "geo": "country_code",
    "time_period": "year",
    "obs_value": "energy_dependency"
})

energy_clean["year"] = pd.to_numeric(energy_clean["year"], errors="coerce")
energy_clean["energy_dependency"] = pd.to_numeric(energy_clean["energy_dependency"], errors="coerce")

energy_clean = energy_clean[["country_code", "year", "energy_dependency"]].drop_duplicates()

energy_clean.head()

,country_code,year,energy_dependency
0,Albania,2015,12.593
1,Albania,2016,20.128
2,Albania,2017,38.126
3,Albania,2018,20.967
4,Albania,2019,31.546


**TRABAJAMOS CON EL EXCEL DE IMPUESTOS**

In [22]:
##Ahora cargamos el Excel de impuestos
tax_xls = pd.ExcelFile(tax_file)

tax_xls.sheet_names

['VAT', 'Excise duties', 'Other Indirect Taxes']

In [ ]:
##Esto nos muestra las hojas dentro del Excel

In [26]:
tax_raw = pd.read_excel(tax_file, sheet_name=tax_xls.sheet_names[0], header=None)

tax_raw.head(10)

,0,1,2,3,4,5,6,7
0,NaN,NaN,"VAT (Value added tax), %",NaN,NaN,NaN,NaN,NaN
1,CTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Since:,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel
3,NaN,NaN,1000 l,1000 l,1000 l,t,t,1000 l
4,AT_,1995-01-01 00:00:00,20,20,20,20,NaN,NaN
5,BE_,1996-01-01 00:00:00,21,21,21,21,NaN,21
6,BG_,1999-01-01 00:00:00,20,20,20,NaN,20,20
7,CY_,2014-01-13 00:00:00,19,19,19,19,NaN,NaN
8,CZ_,2013-01-01 00:00:00,21,21,21,21,19,21
9,DE_,2021-01-01 00:00:00,19,19,19,19,NaN,19


In [33]:
##Ahora realizamos la limpieza básica


# Cargar el Excel SIN cabecera automática
tax_raw = pd.read_excel(
    tax_file,
    sheet_name=tax_xls.sheet_names[0],
    header=None
)

# En este Excel:
# Fila 0 -> título general: VAT (Value added tax), %
# Fila 1 -> CTR
# Fila 2 -> nombres de productos
# Fila 3 -> unidades
# Fila 4 en adelante -> datos reales

product_names = tax_raw.iloc[2].tolist()
units = tax_raw.iloc[3].tolist()

new_columns = []

for i in range(len(product_names)):
    if i == 0:
        new_columns.append("country_code")
    elif i == 1:
        new_columns.append("date")
    else:
        product = str(product_names[i]).strip()
        unit = str(units[i]).strip()
        new_columns.append(f"{product}__{unit}")

# Nos quedamos solo con las filas de datos reales
tax_clean_wide = tax_raw.iloc[4:].copy()
tax_clean_wide.columns = new_columns

# Limpiar código de país
tax_clean_wide["country_code"] = (
    tax_clean_wide["country_code"]
    .astype(str)
    .str.strip()
    .str.replace("_", "", regex=False)
)

# Convertir fecha
tax_clean_wide["date"] = pd.to_datetime(tax_clean_wide["date"], errors="coerce")
tax_clean_wide["year"] = tax_clean_wide["date"].dt.year

# Mapa de países 2 letras -> 3 letras
eu_2_to_3 = {
    "AT": "AUT", "BE": "BEL", "BG": "BGR", "HR": "HRV",
    "CY": "CYP", "CZ": "CZE", "DK": "DNK", "EE": "EST",
    "FI": "FIN", "FR": "FRA", "DE": "DEU", "EL": "GRC",
    "GR": "GRC", "HU": "HUN", "IE": "IRL", "IT": "ITA",
    "LV": "LVA", "LT": "LTU", "LU": "LUX", "MT": "MLT",
    "NL": "NLD", "PL": "POL", "PT": "PRT", "RO": "ROU",
    "SK": "SVK", "SI": "SVN", "ES": "ESP", "SE": "SWE"
}

tax_clean_wide["country_code_2"] = tax_clean_wide["country_code"]
tax_clean_wide["country_code"] = tax_clean_wide["country_code_2"].map(eu_2_to_3)

# Pasar de formato ancho a formato largo
id_cols = ["country_code", "country_code_2", "date", "year"]
value_cols = [c for c in tax_clean_wide.columns if c not in id_cols]

tax_long = tax_clean_wide.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="product_unit",
    value_name="tax_value"
)

# Separar producto y unidad
tax_long[["product", "unit"]] = tax_long["product_unit"].str.split("__", expand=True)
tax_long = tax_long.drop(columns=["product_unit"])

# Convertir impuesto a número
tax_long["tax_value"] = pd.to_numeric(tax_long["tax_value"], errors="coerce")

# Eliminar nulos y duplicados
entrada_tax = len(tax_long)

tax_long = tax_long.dropna(subset=["country_code", "year", "product", "tax_value"])
tax_long = tax_long.drop_duplicates()

# Convertir a €/litro cuando la unidad sea 1000 l
tax_long["tax_eur_litre"] = np.where(
    tax_long["unit"].astype(str).str.lower().str.contains("1000", na=False),
    tax_long["tax_value"] / 1000,
    np.nan
)

# Crear producto simplificado
def simplify_tax_product(product):
    product = str(product).lower()

    if "euro-super" in product or "super 95" in product:
        return "gasoline"
    elif "gas oil automobile" in product or "diesel" in product:
        return "diesel"
    else:
        return "other"

tax_long["fuel_type"] = tax_long["product"].apply(simplify_tax_product)

# Dataset final de impuestos
tax_final = tax_long[[
    "country_code",
    "country_code_2",
    "date",
    "year",
    "product",
    "fuel_type",
    "unit",
    "tax_value",
    "tax_eur_litre"
]].copy()

# Guardar tracking si la función existe
try:
    add_tracking(
        fase="limpieza_transformacion",
        dataset="impuestos_oil_bulletin",
        registros_entrada=entrada_tax,
        registros_salida=len(tax_final),
        eliminados=entrada_tax - len(tax_final),
        motivo="Reconstrucción de cabecera, transformación a formato largo, limpieza de nulos y conversión de impuestos a €/litro cuando la unidad es 1000 l"
    )
except NameError:
    print("Aviso: add_tracking todavía no está definida. No se ha guardado tracking de impuestos.")

tax_final.head(20)

,country_code,country_code_2,date,year,product,fuel_type,unit,tax_value,tax_eur_litre
0,AUT,AT,1995-01-01,1995.0,Euro-super 95 (I),gasoline,1000 l,20.0,0.0200
1,BEL,BE,1996-01-01,1996.0,Euro-super 95 (I),gasoline,1000 l,21.0,0.0210
2,BGR,BG,1999-01-01,1999.0,Euro-super 95 (I),gasoline,1000 l,20.0,0.0200
3,CYP,CY,2014-01-13,2014.0,Euro-super 95 (I),gasoline,1000 l,19.0,0.0190
4,CZE,CZ,2013-01-01,2013.0,Euro-super 95 (I),gasoline,1000 l,21.0,0.0210
5,DEU,DE,2021-01-01,2021.0,Euro-super 95 (I),gasoline,1000 l,19.0,0.0190
6,DNK,DK,1995-01-01,1995.0,Euro-super 95 (I),gasoline,1000 l,25.0,0.0250
7,EST,EE,2026-01-01,2026.0,Euro-super 95 (I),gasoline,1000 l,22.0,0.0220
8,ESP,ES,2026-03-22,2026.0,Euro-super 95 (I),gasoline,1000 l,10.0,0.0100
9,FIN,FI,2026-01-01,2026.0,Euro-super 95 (I),gasoline,1000 l,25.5,0.0255


In [34]:
tax_clean_wide.head()

,country_code,date,Euro-super 95 (I)__1000 l,Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l,Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l,Fuel oil - Schweres Heizöl (III) Soufre__t,Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,GPL pour moteur LPG motor fuel__1000 l,year,country_code_2
4,AUT,1995-01-01,20,20,20,20,NaN,NaN,1995.0,AT
5,BEL,1996-01-01,21,21,21,21,NaN,21,1996.0,BE
6,BGR,1999-01-01,20,20,20,NaN,20,20,1999.0,BG
7,CYP,2014-01-13,19,19,19,19,NaN,NaN,2014.0,CY
8,CZE,2013-01-01,21,21,21,21,19,21,2013.0,CZ


In [35]:
tax_final[["product", "fuel_type", "unit"]].drop_duplicates()

,product,fuel_type,unit
0,Euro-super 95 (I),gasoline,1000 l
32,Gas oil automobile Automotive gas oil Dieselkr...,diesel,1000 l
64,Gas oil de chauffage Heating gas oil Heizöl (II),other,1000 l
96,Fuel oil - Schweres Heizöl (III) Soufre,other,t
130,Fuel oil -Schweres Heizöl (III) Soufre > 1% Su...,other,t
161,GPL pour moteur LPG motor fuel,other,1000 l


In [36]:
## Nos quedamos solo con gasolina y diésel
tax_fuel = tax_final[
    tax_final["fuel_type"].isin(["gasoline", "diesel"])
].copy()

tax_fuel.head(20)
##Esto elimina los productos que no vamos a usar, como fuel oil, GPL, etc.

,country_code,country_code_2,date,year,product,fuel_type,unit,tax_value,tax_eur_litre
0,AUT,AT,1995-01-01,1995.0,Euro-super 95 (I),gasoline,1000 l,20.0,0.0200
1,BEL,BE,1996-01-01,1996.0,Euro-super 95 (I),gasoline,1000 l,21.0,0.0210
2,BGR,BG,1999-01-01,1999.0,Euro-super 95 (I),gasoline,1000 l,20.0,0.0200
3,CYP,CY,2014-01-13,2014.0,Euro-super 95 (I),gasoline,1000 l,19.0,0.0190
4,CZE,CZ,2013-01-01,2013.0,Euro-super 95 (I),gasoline,1000 l,21.0,0.0210
5,DEU,DE,2021-01-01,2021.0,Euro-super 95 (I),gasoline,1000 l,19.0,0.0190
6,DNK,DK,1995-01-01,1995.0,Euro-super 95 (I),gasoline,1000 l,25.0,0.0250
7,EST,EE,2026-01-01,2026.0,Euro-super 95 (I),gasoline,1000 l,22.0,0.0220
8,ESP,ES,2026-03-22,2026.0,Euro-super 95 (I),gasoline,1000 l,10.0,0.0100
9,FIN,FI,2026-01-01,2026.0,Euro-super 95 (I),gasoline,1000 l,25.5,0.0255


In [37]:
##Comprobación para saber si está bien
tax_fuel[["country_code", "country_code_2", "year", "product", "fuel_type", "unit", "tax_value", "tax_eur_litre"]].head(20)

,country_code,country_code_2,year,product,fuel_type,unit,tax_value,tax_eur_litre
0,AUT,AT,1995.0,Euro-super 95 (I),gasoline,1000 l,20.0,0.0200
1,BEL,BE,1996.0,Euro-super 95 (I),gasoline,1000 l,21.0,0.0210
2,BGR,BG,1999.0,Euro-super 95 (I),gasoline,1000 l,20.0,0.0200
3,CYP,CY,2014.0,Euro-super 95 (I),gasoline,1000 l,19.0,0.0190
4,CZE,CZ,2013.0,Euro-super 95 (I),gasoline,1000 l,21.0,0.0210
5,DEU,DE,2021.0,Euro-super 95 (I),gasoline,1000 l,19.0,0.0190
6,DNK,DK,1995.0,Euro-super 95 (I),gasoline,1000 l,25.0,0.0250
7,EST,EE,2026.0,Euro-super 95 (I),gasoline,1000 l,22.0,0.0220
8,ESP,ES,2026.0,Euro-super 95 (I),gasoline,1000 l,10.0,0.0100
9,FIN,FI,2026.0,Euro-super 95 (I),gasoline,1000 l,25.5,0.0255


In [41]:
##Compruebo duplicados
tax_fuel.duplicated(subset=["country_code", "year", "fuel_type"]).sum()

np.int64(0)

In [43]:
##Cambio el año a número entero
tax_fuel["year"] = tax_fuel["year"].astype("Int64")

In [45]:
tax_fuel[["country_code", "year", "fuel_type", "tax_value", "tax_eur_litre"]].head(20)

,country_code,year,fuel_type,tax_value,tax_eur_litre
0,AUT,1995,gasoline,20.0,0.0200
1,BEL,1996,gasoline,21.0,0.0210
2,BGR,1999,gasoline,20.0,0.0200
3,CYP,2014,gasoline,19.0,0.0190
4,CZE,2013,gasoline,21.0,0.0210
5,DEU,2021,gasoline,19.0,0.0190
6,DNK,1995,gasoline,25.0,0.0250
7,EST,2026,gasoline,22.0,0.0220
8,ESP,2026,gasoline,10.0,0.0100
9,FIN,2026,gasoline,25.5,0.0255


In [46]:
tax_fuel.to_csv("data/processed/impuestos_procesados.csv", index=False, encoding="utf-8")

print("Archivo guardado: data/processed/impuestos_procesados.csv")

Archivo guardado: data/processed/impuestos_procesados.csv


In [47]:
!ls -lah data/processed

total 16K
drwxr-xr-x 2 root root 4.0K May 12 09:41 .
drwxr-xr-x 4 root root 4.0K May 12 08:58 ..
-rw-r--r-- 1 root root 4.8K May 12 09:45 impuestos_procesados.csv


**PROCESAMIENTO DEL WEEKLY OIL BULLETIN PRICES**

In [54]:
prices_xls = pd.ExcelFile(prices_file)

print("Hojas del Excel de precios:")
print(prices_xls.sheet_names)

# Usamos la hoja "Prices with taxes", que es la primera
prices_raw = pd.read_excel(
    prices_file,
    sheet_name="Prices with taxes",
    header=None
)

print("Dimensiones originales:", prices_raw.shape)

# En este Excel se ha detectado que la fila 2 funciona como fila de cabecera/unidades
unit_row = 2
print("Fila usada como fila de cabecera/unidades:", unit_row)

# Tomamos filas superiores como cabeceras jerárquicas
header_start = max(0, unit_row - 2)
header_rows = prices_raw.iloc[header_start:unit_row + 1].copy()

# Rellenar celdas combinadas hacia la derecha
header_rows = header_rows.ffill(axis=1)

# Datos reales
prices_data = prices_raw.iloc[unit_row + 1:].copy()

# Construcción de nombres de columnas SIN duplicados
new_columns = []

for j in range(prices_raw.shape[1]):
    parts = []

    for i in header_rows.index:
        value = header_rows.loc[i, j]

        if pd.notna(value):
            value = str(value).strip()

            if value != "" and value.lower() != "nan":
                parts.append(value)

    # Eliminar partes repetidas manteniendo orden
    clean_parts = []

    for p in parts:
        if p not in clean_parts:
            clean_parts.append(p)

    col_name = "__".join(clean_parts)

    # Solo la primera columna será date
    if j == 0:
        col_name = "date"

    # Si por casualidad otra columna contiene Date, le damos nombre único
    elif "date" in col_name.lower():
        col_name = f"date_info_{j}"

    # Si queda vacío, le ponemos nombre técnico
    elif col_name == "":
        col_name = f"column_{j}"

    new_columns.append(col_name)

# Asegurar que no hay nombres duplicados
seen = {}
unique_columns = []

for col in new_columns:
    if col not in seen:
        seen[col] = 0
        unique_columns.append(col)
    else:
        seen[col] += 1
        unique_columns.append(f"{col}_{seen[col]}")

prices_data.columns = unique_columns

print("Primeras columnas generadas:")
print(prices_data.columns.tolist()[:20])

print("Número de columnas duplicadas:")
print(pd.Series(prices_data.columns).duplicated().sum())

# Convertimos solo la primera columna date
prices_data["date"] = pd.to_datetime(prices_data["date"], errors="coerce")

# Eliminamos filas sin fecha
entrada_prices = len(prices_data)

prices_data = prices_data.dropna(subset=["date"]).copy()

# Eliminamos duplicados
prices_data = prices_data.drop_duplicates()

print("Filas antes:", entrada_prices)
print("Filas después de limpiar fechas y duplicados:", len(prices_data))

prices_data.head()

Hojas del Excel de precios:
['Prices with taxes', 'Prices wo taxes', 'Consumption', 'VAT', 'Excise duties', 'Excise duties - components', 'Other Indirect Taxes']
Dimensiones originales: (1092, 226)
Fila usada como fila de cabecera/unidades: 2
Primeras columnas generadas:
['date', 'date_info_1', 'EU_price_with_tax_euro95__Euro-super 95  (I)__1000 l', 'EU_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l', 'EU_price_with_tax_heEUing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l', 'EU_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t', 'EU_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t', 'EU_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l', 'CTR__GPL pour moteur LPG motor fuel__1000 l', 'EUR_price_with_tax_euro95__Euro-super 95  (I)__1000 l', 'EUR_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l', 'EUR_price_w

,date,date_info_1,EU_price_with_tax_euro95__Euro-super 95 (I)__1000 l,EU_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l,EU_price_with_tax_heEUing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l,EU_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t,EU_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,EU_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l,CTR__GPL pour moteur LPG motor fuel__1000 l,EUR_price_with_tax_euro95__Euro-super 95 (I)__1000 l,...,SK_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,SK_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l,CTR__GPL pour moteur LPG motor fuel__1000 l_28,UK_exchange_rate__GPL pour moteur LPG motor fuel__1000 l,UK_price_with_tax_euro95__Euro-super 95 (I)__1000 l,UK_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l,UK_price_with_tax_heUKing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l,UK_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t,UK_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,UK_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l
3,2026-04-27,EU_,1841.515072,1974.343876,1421.360167,766.447376,595.076998,934.959553,EUR_,1899.994139,...,NaN,901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-20,EU_,1827.449889,2007.925502,1463.983752,782.251889,588.093422,933.52501,EUR_,1887.468271,...,NaN,901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-04-13,EU_,1853.723723,2099.719327,1544.892508,731.107881,631.19693,922.967746,EUR_,1915.431289,...,NaN,852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-04-06,EU_,1878.734029,2112.274655,1648.816741,741.774199,659.757575,889.877673,EUR_,1945.947463,...,NaN,752,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-03-30,EU_,1871.508517,2075.661431,1452.484739,751.346501,642.248671,841.190321,EUR_,1907.693838,...,NaN,737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
# PREPARAR PRECIOS: EL PAÍS ESTÁ EN EL NOMBRE DE LA COLUMNA

# Si existe date_info_1, la eliminamos porque no nos sirve para el dataset final
if "date_info_1" in prices_data.columns:
    prices_data = prices_data.drop(columns=["date_info_1"])

print("Filas de prices_data:", len(prices_data))
print("Columnas de prices_data:", len(prices_data.columns))

prices_data.head()

Filas de prices_data: 1064
Columnas de prices_data: 225


,date,EU_price_with_tax_euro95__Euro-super 95 (I)__1000 l,EU_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l,EU_price_with_tax_heEUing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l,EU_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t,EU_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,EU_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l,CTR__GPL pour moteur LPG motor fuel__1000 l,EUR_price_with_tax_euro95__Euro-super 95 (I)__1000 l,EUR_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l,...,SK_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,SK_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l,CTR__GPL pour moteur LPG motor fuel__1000 l_28,UK_exchange_rate__GPL pour moteur LPG motor fuel__1000 l,UK_price_with_tax_euro95__Euro-super 95 (I)__1000 l,UK_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l,UK_price_with_tax_heUKing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l,UK_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t,UK_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t,UK_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l
3,2026-04-27,1841.515072,1974.343876,1421.360167,766.447376,595.076998,934.959553,EUR_,1899.994139,2031.158103,...,NaN,901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-20,1827.449889,2007.925502,1463.983752,782.251889,588.093422,933.52501,EUR_,1887.468271,2067.039967,...,NaN,901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-04-13,1853.723723,2099.719327,1544.892508,731.107881,631.19693,922.967746,EUR_,1915.431289,2154.223347,...,NaN,852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-04-06,1878.734029,2112.274655,1648.816741,741.774199,659.757575,889.877673,EUR_,1945.947463,2158.252964,...,NaN,752,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-03-30,1871.508517,2075.661431,1452.484739,751.346501,642.248671,841.190321,EUR_,1907.693838,2090.282212,...,NaN,737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
# Comprobamos ejemplos de columnas de países reales
[c for c in prices_data.columns if c.startswith("ES_")][:10]

['ES_price_with_tax_euro95__Euro-super 95  (I)__1000 l',
 'ES_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l',
 'ES_price_with_tax_heESing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l',
 'ES_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t',
 'ES_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t',
 'ES_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l']

In [57]:
[c for c in prices_data.columns if c.startswith("FR_")][:10]

['FR_price_with_tax_euro95__Euro-super 95  (I)__1000 l',
 'FR_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l',
 'FR_price_with_tax_heFRing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l',
 'FR_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t',
 'FR_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t',
 'FR_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l']

In [58]:
[c for c in prices_data.columns if c.startswith("DE_")][:10]

['DE_price_with_tax_euro95__Euro-super 95  (I)__1000 l',
 'DE_price_with_tax_diesel__Gas oil automobile Automotive gas oil Dieselkraftstoff (I)__1000 l',
 'DE_price_with_tax_heDEing_oil__Gas oil de chauffage Heating gas oil Heizöl (II)__1000 l',
 'DE_price_with_tax_fuel_oil_1__Fuel oil - Schweres Heizöl (III) Soufre__t',
 'DE_price_with_tax_fuel_oil_2__Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%__t',
 'DE_price_with_tax_LPG__GPL pour moteur LPG motor fuel__1000 l']

In [59]:
##Precios a formato largo

id_cols = ["date"]

value_cols = [c for c in prices_data.columns if c not in id_cols]

prices_long = prices_data.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="raw_product",
    value_name="price_original"
)

# Convertir precio a número
prices_long["price_original"] = pd.to_numeric(
    prices_long["price_original"],
    errors="coerce"
)

# Crear año
prices_long["year"] = prices_long["date"].dt.year.astype("Int64")

# Eliminar nulos y precios no válidos
entrada_prices_long = len(prices_long)

prices_long = prices_long.dropna(
    subset=["date", "price_original", "year"]
).copy()

prices_long = prices_long[prices_long["price_original"] > 0].copy()

print("Filas antes:", entrada_prices_long)
print("Filas después:", len(prices_long))

prices_long.head(20)

Filas antes: 238336
Filas después: 155847


,date,raw_product,price_original,year
0,2026-04-27,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1841.515072,2026
1,2026-04-20,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1827.449889,2026
2,2026-04-13,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1853.723723,2026
3,2026-04-06,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1878.734029,2026
4,2026-03-30,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1871.508517,2026
5,2026-03-23,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1893.491227,2026
6,2026-03-16,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1837.926054,2026
7,2026-03-09,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1774.084779,2026
8,2026-03-02,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1664.651018,2026
9,2026-02-23,EU_price_with_tax_euro95__Euro-super 95 (I)__...,1637.231143,2026


In [60]:
##Detección de País, Combustible y Unidad desde Raw_Product

eu_2_to_3 = {
    "AT": "AUT", "BE": "BEL", "BG": "BGR", "HR": "HRV",
    "CY": "CYP", "CZ": "CZE", "DK": "DNK", "EE": "EST",
    "FI": "FIN", "FR": "FRA", "DE": "DEU", "EL": "GRC",
    "GR": "GRC", "HU": "HUN", "IE": "IRL", "IT": "ITA",
    "LV": "LVA", "LT": "LTU", "LU": "LUX", "MT": "MLT",
    "NL": "NLD", "PL": "POL", "PT": "PRT", "RO": "ROU",
    "SK": "SVK", "SI": "SVN", "ES": "ESP", "SE": "SWE"
}

def detect_country_code_2_from_column(text):
    text = str(text).strip()
    code_2 = text.split("_")[0].upper()

    if code_2 in eu_2_to_3:
        return code_2
    else:
        return np.nan

def detect_country_from_column(text):
    text = str(text).strip()
    code_2 = text.split("_")[0].upper()

    return eu_2_to_3.get(code_2, np.nan)

def detect_fuel_type(text):
    text = str(text).lower()

    if "euro95" in text or "euro-super" in text or "super_95" in text or "super 95" in text:
        return "gasoline"
    elif "diesel" in text or "gas_oil_automobile" in text or "automotive_gas_oil" in text:
        return "diesel"
    else:
        return "other"

def detect_unit(text):
    text = str(text).lower()

    if "1000 l" in text or "1000_l" in text or "1000l" in text:
        return "1000 l"
    elif "_t" in text or text.endswith("_t") or "soufre__t" in text:
        return "t"
    else:
        return np.nan

prices_long["country_code_2"] = prices_long["raw_product"].apply(detect_country_code_2_from_column)
prices_long["country_code"] = prices_long["raw_product"].apply(detect_country_from_column)
prices_long["fuel_type"] = prices_long["raw_product"].apply(detect_fuel_type)
prices_long["unit"] = prices_long["raw_product"].apply(detect_unit)

prices_long[[
    "raw_product",
    "country_code_2",
    "country_code",
    "fuel_type",
    "unit"
]].drop_duplicates().head(80)

,raw_product,country_code_2,country_code,fuel_type,unit
0,EU_price_with_tax_euro95__Euro-super 95 (I)__...,NaN,NaN,gasoline,1000 l
1064,EU_price_with_tax_diesel__Gas oil automobile A...,NaN,NaN,diesel,1000 l
2128,EU_price_with_tax_heEUing_oil__Gas oil de chau...,NaN,NaN,other,1000 l
3192,EU_price_with_tax_fuel_oil_1__Fuel oil - Schwe...,NaN,NaN,other,t
4256,EU_price_with_tax_fuel_oil_2__Fuel oil -Schwer...,NaN,NaN,other,t
...,...,...,...,...,...
107328,GR_price_with_tax_fuel_oil_2__Fuel oil -Schwer...,GR,GRC,other,t
109592,HR_exchange_rate__GPL pour moteur LPG motor fu...,HR,HRV,other,1000 l
110656,HR_price_with_tax_euro95__Euro-super 95 (I)__...,HR,HRV,gasoline,1000 l
111720,HR_price_with_tax_diesel__Gas oil automobile A...,HR,HRV,diesel,1000 l


In [62]:
# Limpieza final de Precios

entrada_prices_clean = len(prices_long)

prices_clean = prices_long.copy()

# Nos quedamos solo con países reales
prices_clean = prices_clean.dropna(subset=["country_code"]).copy()

# Nos quedamos solo con gasolina y diésel
prices_clean = prices_clean[
    prices_clean["fuel_type"].isin(["gasoline", "diesel"])
].copy()

# Convertimos a €/litro.
# En este Excel los precios vienen en €/1000 litros.
prices_clean["price_eur_litre"] = prices_clean["price_original"] / 1000

# Eliminamos precios raros
prices_clean = prices_clean[
    (prices_clean["price_eur_litre"] > 0) &
    (prices_clean["price_eur_litre"] < 5)
].copy()

# Eliminamos duplicados
prices_clean = prices_clean.drop_duplicates()

# Seleccionamos columnas finales de precios
prices_clean = prices_clean[[
    "date",
    "year",
    "country_code",
    "country_code_2",
    "fuel_type",
    "price_original",
    "price_eur_litre",
    "unit",
    "raw_product"
]].copy()

add_tracking(
    fase="limpieza_transformacion",
    dataset="weekly_oil_bulletin_prices",
    registros_entrada=entrada_prices_clean,
    registros_salida=len(prices_clean),
    eliminados=entrada_prices_clean - len(prices_clean),
    motivo="Transformación a formato largo, detección de país desde columnas, selección de gasolina y diésel, conversión de precios de €/1000 litros a €/litro y eliminación de valores no válidos"
)

prices_clean.head(20)

,date,year,country_code,country_code_2,fuel_type,price_original,price_eur_litre,unit,raw_product
14896,2026-04-27,2026,AUT,AT,gasoline,1712.0,1.712,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14897,2026-04-20,2026,AUT,AT,gasoline,1691.0,1.691,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14898,2026-04-13,2026,AUT,AT,gasoline,1718.0,1.718,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14899,2026-04-06,2026,AUT,AT,gasoline,1787.0,1.787,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14900,2026-03-30,2026,AUT,AT,gasoline,1879.0,1.879,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14901,2026-03-23,2026,AUT,AT,gasoline,1841.0,1.841,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14902,2026-03-16,2026,AUT,AT,gasoline,1743.0,1.743,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14903,2026-03-09,2026,AUT,AT,gasoline,1708.0,1.708,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14904,2026-03-02,2026,AUT,AT,gasoline,1515.0,1.515,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...
14905,2026-02-23,2026,AUT,AT,gasoline,1509.0,1.509,1000 l,AT_price_with_tax_euro95__Euro-super 95 (I)__...


In [63]:
##Comprobación final de precios
print("Filas de precios limpias:", len(prices_clean))
print("Países detectados:", prices_clean["country_code"].nunique())

print("\nTipos de combustible:")
print(prices_clean["fuel_type"].value_counts())

print("\nRango de fechas:")
print(prices_clean["date"].min(), "→", prices_clean["date"].max())

prices_clean[[
    "date",
    "year",
    "country_code",
    "country_code_2",
    "fuel_type",
    "price_original",
    "price_eur_litre"
]].head(20)

Filas de precios limpias: 56011
Países detectados: 27

Tipos de combustible:
fuel_type
diesel      28006
gasoline    28005
Name: count, dtype: int64

Rango de fechas:
2005-01-03 00:00:00 → 2026-04-27 00:00:00


,date,year,country_code,country_code_2,fuel_type,price_original,price_eur_litre
14896,2026-04-27,2026,AUT,AT,gasoline,1712.0,1.712
14897,2026-04-20,2026,AUT,AT,gasoline,1691.0,1.691
14898,2026-04-13,2026,AUT,AT,gasoline,1718.0,1.718
14899,2026-04-06,2026,AUT,AT,gasoline,1787.0,1.787
14900,2026-03-30,2026,AUT,AT,gasoline,1879.0,1.879
14901,2026-03-23,2026,AUT,AT,gasoline,1841.0,1.841
14902,2026-03-16,2026,AUT,AT,gasoline,1743.0,1.743
14903,2026-03-09,2026,AUT,AT,gasoline,1708.0,1.708
14904,2026-03-02,2026,AUT,AT,gasoline,1515.0,1.515
14905,2026-02-23,2026,AUT,AT,gasoline,1509.0,1.509


**PREPARAMOS EL PIB PER CÁPITA**

In [64]:
gdp_ready = gdp_long[[
    "country_code",
    "country_name",
    "year",
    "gdp_per_capita"
]].copy()

gdp_ready["year"] = gdp_ready["year"].astype("Int64")

gdp_ready = gdp_ready.drop_duplicates(subset=["country_code", "year"])

print("Filas PIB preparadas:", len(gdp_ready))
gdp_ready.head()

Filas PIB preparadas: 17556


,country_code,country_name,year,gdp_per_capita
0,ABW,Aruba,1960,NaN
1,AFE,Africa Eastern and Southern,1960,186.089204
2,AFG,Afghanistan,1960,NaN
3,AFW,Africa Western and Central,1960,121.936832
4,AGO,Angola,1960,NaN


In [73]:
# Preparar dependencia energética

energy_ready = energy_clean.copy()

# Limpiamos nombres
energy_ready["country_name_energy"] = (
    energy_ready["country_code"]
    .astype(str)
    .str.strip()
)

# Mapa de nombres de país a código ISO3
country_name_to_iso3 = {
    "Austria": "AUT",
    "Belgium": "BEL",
    "Bulgaria": "BGR",
    "Croatia": "HRV",
    "Cyprus": "CYP",
    "Czechia": "CZE",
    "Czech Republic": "CZE",
    "Denmark": "DNK",
    "Estonia": "EST",
    "Finland": "FIN",
    "France": "FRA",
    "Germany": "DEU",
    "Greece": "GRC",
    "Hungary": "HUN",
    "Ireland": "IRL",
    "Italy": "ITA",
    "Latvia": "LVA",
    "Lithuania": "LTU",
    "Luxembourg": "LUX",
    "Malta": "MLT",
    "Netherlands": "NLD",
    "Poland": "POL",
    "Portugal": "PRT",
    "Romania": "ROU",
    "Slovakia": "SVK",
    "Slovenia": "SVN",
    "Spain": "ESP",
    "Sweden": "SWE"
}

# Convertimos nombre de país a código ISO3
energy_ready["country_code"] = energy_ready["country_name_energy"].map(country_name_to_iso3)

# Convertimos año y valor
energy_ready["year"] = pd.to_numeric(energy_ready["year"], errors="coerce").astype("Int64")

energy_ready["energy_dependency"] = pd.to_numeric(
    energy_ready["energy_dependency"],
    errors="coerce"
)

# Nos quedamos solo con países que están en nuestro dataset europeo
energy_ready = energy_ready.dropna(subset=["country_code", "year", "energy_dependency"]).copy()

energy_ready = energy_ready[[
    "country_code",
    "year",
    "energy_dependency"
]].copy()

# Evitamos duplicados por país y año
energy_ready = energy_ready.drop_duplicates(subset=["country_code", "year"])

print("Filas energía preparadas:", len(energy_ready))
print("Años energía:", energy_ready["year"].min(), "→", energy_ready["year"].max())
print("Países energía:", energy_ready["country_code"].nunique())

energy_ready.head(20)

Filas energía preparadas: 270
Años energía: 2015 → 2024
Países energía: 27


,country_code,year,energy_dependency
10,AUT,2015,60.379
11,AUT,2016,62.102
12,AUT,2017,63.930
13,AUT,2018,64.223
14,AUT,2019,71.601
15,AUT,2020,58.281
16,AUT,2021,51.952
17,AUT,2022,73.986
18,AUT,2023,60.317
19,AUT,2024,53.325


La tabla tax_fuel tiene el año en el que empieza a aplicar cada impuesto. Para poder unirlo con todos los años de precios, vamos a expandirlo y arrastrar el último impuesto conocido hacia delante

In [74]:
# Expandir impuestos por año

tax_fuel["year"] = tax_fuel["year"].astype("Int64")

# Año mínimo y máximo de precios
min_year_prices = int(prices_clean["year"].min())
max_year_prices = int(prices_clean["year"].max())

# Año mínimo de impuestos
min_year_tax = int(tax_fuel["year"].min())

# Hay que empezar desde el año mínimo de impuestos,
# no desde el año mínimo de precios, para poder arrastrar valores antiguos.
min_year_expand = min(min_year_prices, min_year_tax)

all_years = pd.DataFrame({
    "year": range(min_year_expand, max_year_prices + 1)
})

countries = tax_fuel["country_code"].dropna().unique()
fuels = tax_fuel["fuel_type"].dropna().unique()

base_tax = pd.MultiIndex.from_product(
    [countries, fuels, all_years["year"]],
    names=["country_code", "fuel_type", "year"]
).to_frame(index=False)

tax_yearly = base_tax.merge(
    tax_fuel[[
        "country_code",
        "fuel_type",
        "year",
        "tax_value",
        "tax_eur_litre",
        "unit",
        "product"
    ]],
    on=["country_code", "fuel_type", "year"],
    how="left"
)

tax_yearly = tax_yearly.sort_values(["country_code", "fuel_type", "year"])

# Arrastrar hacia delante el último impuesto conocido
tax_yearly[["tax_value", "tax_eur_litre", "unit", "product"]] = (
    tax_yearly
    .groupby(["country_code", "fuel_type"])[["tax_value", "tax_eur_litre", "unit", "product"]]
    .ffill()
)

# Ahora sí, nos quedamos solo con los años que existen en precios
tax_yearly = tax_yearly[
    (tax_yearly["year"] >= min_year_prices) &
    (tax_yearly["year"] <= max_year_prices)
].copy()

# Eliminamos los casos donde nunca haya habido impuesto conocido
tax_yearly = tax_yearly.dropna(subset=["tax_eur_litre"]).copy()

print("Año mínimo precios:", min_year_prices)
print("Año máximo precios:", max_year_prices)
print("Año mínimo impuestos:", min_year_tax)
print("Filas impuestos anuales:", len(tax_yearly))

tax_yearly.head(20)

Año mínimo precios: 2005
Año máximo precios: 2026
Año mínimo impuestos: 1995
Filas impuestos anuales: 610


,country_code,fuel_type,year,tax_value,tax_eur_litre,unit,product
42,AUT,diesel,2005,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
43,AUT,diesel,2006,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
44,AUT,diesel,2007,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
45,AUT,diesel,2008,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
46,AUT,diesel,2009,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
47,AUT,diesel,2010,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
48,AUT,diesel,2011,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
49,AUT,diesel,2012,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
50,AUT,diesel,2013,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
51,AUT,diesel,2014,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...


In [77]:
energy_clean.head(20)

,country_code,year,energy_dependency
0,Albania,2015,12.593
1,Albania,2016,20.128
2,Albania,2017,38.126
3,Albania,2018,20.967
4,Albania,2019,31.546
5,Albania,2020,35.822
6,Albania,2021,23.765
7,Albania,2022,31.467
8,Albania,2023,21.845
9,Albania,2024,27.138


In [81]:
sorted(energy_ready["country_code"].dropna().unique())

['AUT',
 'BEL',
 'BGR',
 'CYP',
 'CZE',
 'DEU',
 'DNK',
 'ESP',
 'EST',
 'FIN',
 'FRA',
 'GRC',
 'HRV',
 'HUN',
 'IRL',
 'ITA',
 'LTU',
 'LUX',
 'LVA',
 'MLT',
 'NLD',
 'POL',
 'PRT',
 'ROU',
 'SVK',
 'SVN',
 'SWE']

In [84]:
# La dependencia energética empieza más tarde que la serie de precios.
# Por eso existen nulos en energy_dependency para años anteriores a la disponibilidad del indicador.

print("Años en precios:", dataset["year"].min(), "→", dataset["year"].max())
print("Años en dependencia energética:", energy_ready["year"].min(), "→", energy_ready["year"].max())

Años en precios: 2005 → 2026
Años en dependencia energética: 2015 → 2024


In [85]:
##Revisión de nulos en los impuestos

print("Nulos de impuestos por tipo de combustible:")
print(
    dataset
    .groupby("fuel_type")[["tax_value", "tax_eur_litre"]]
    .apply(lambda x: (x.isna().mean() * 100).round(2))
)

print("\nPaíses en precios:")
print(sorted(prices_clean["country_code"].dropna().unique()))

print("\nPaíses en impuestos:")
print(sorted(tax_yearly["country_code"].dropna().unique()))

Nulos de impuestos por tipo de combustible:
           tax_value  tax_eur_litre
fuel_type                          
diesel         49.34          49.34
gasoline       49.34          49.34

Países en precios:
['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']

Países en impuestos:
['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']


In [86]:
print("Años en precios:", prices_clean["year"].min(), "→", prices_clean["year"].max())
print("Años en impuestos:", tax_yearly["year"].min(), "→", tax_yearly["year"].max())

Años en precios: 2005 → 2026
Años en impuestos: 2005 → 2026


In [87]:
# Comprobar nulos de impuestos por combustible

print(
    dataset
    .groupby("fuel_type")[["tax_value", "tax_eur_litre"]]
    .apply(lambda x: (x.isna().mean() * 100).round(2))
)

           tax_value  tax_eur_litre
fuel_type                          
diesel         49.34          49.34
gasoline       49.34          49.34


In [88]:
# Comprobar tax_yearly por combustible

print(tax_yearly["fuel_type"].value_counts())

tax_yearly.groupby("fuel_type")[["tax_value", "tax_eur_litre"]].apply(
    lambda x: (x.isna().mean() * 100).round(2)
)

fuel_type
diesel      305
gasoline    305
Name: count, dtype: int64


,tax_value,tax_eur_litre
fuel_type,,
diesel,0.0,0.0
gasoline,0.0,0.0


In [92]:
# ============================
# REHACER TAX_YEARLY COMPLETO - VERSIÓN FINAL CORREGIDA
# ============================

tax_fuel["year"] = tax_fuel["year"].astype("Int64")

min_year_prices = int(prices_clean["year"].min())
max_year_prices = int(prices_clean["year"].max())
min_year_tax = int(tax_fuel["year"].min())

# Importante:
# empezamos desde el primer año disponible en impuestos,
# no desde el primer año de precios.
# Así podemos arrastrar impuestos antiguos hacia 2005 en adelante.
min_year_expand = min(min_year_tax, min_year_prices)

all_years = list(range(min_year_expand, max_year_prices + 1))

countries = sorted(prices_clean["country_code"].dropna().unique())
fuels = sorted(prices_clean["fuel_type"].dropna().unique())

base_tax = pd.MultiIndex.from_product(
    [countries, fuels, all_years],
    names=["country_code", "fuel_type", "year"]
).to_frame(index=False)

tax_yearly = base_tax.merge(
    tax_fuel[[
        "country_code",
        "fuel_type",
        "year",
        "tax_value",
        "tax_eur_litre",
        "unit",
        "product"
    ]],
    on=["country_code", "fuel_type", "year"],
    how="left"
)

tax_yearly = tax_yearly.sort_values(["country_code", "fuel_type", "year"])

cols_fill = ["tax_value", "tax_eur_litre", "unit", "product"]

# Rellenamos hacia delante para arrastrar impuestos antiguos a años posteriores
tax_yearly[cols_fill] = (
    tax_yearly
    .groupby(["country_code", "fuel_type"])[cols_fill]
    .ffill()
)

# Rellenamos hacia atrás para cubrir años anteriores al primer dato disponible,
# cuando exista al menos un dato para ese país y combustible
tax_yearly[cols_fill] = (
    tax_yearly
    .groupby(["country_code", "fuel_type"])[cols_fill]
    .bfill()
)

# Ahora filtramos solo los años de precios
tax_yearly = tax_yearly[
    (tax_yearly["year"] >= min_year_prices) &
    (tax_yearly["year"] <= max_year_prices)
].copy()

print("Año mínimo precios:", min_year_prices)
print("Año máximo precios:", max_year_prices)
print("Año mínimo impuestos:", min_year_tax)

print("\nFilas esperadas:", len(countries) * len(fuels) * (max_year_prices - min_year_prices + 1))
print("Filas tax_yearly:", len(tax_yearly))

print("\nNulos en tax_yearly:")
print(tax_yearly.isna().sum())

print("\nFilas por combustible:")
print(tax_yearly["fuel_type"].value_counts())

tax_yearly.head(20)

Año mínimo precios: 2005
Año máximo precios: 2026
Año mínimo impuestos: 1995

Filas esperadas: 1188
Filas tax_yearly: 1188

Nulos en tax_yearly:
country_code     0
fuel_type        0
year             0
tax_value        0
tax_eur_litre    0
unit             0
product          0
dtype: int64

Filas por combustible:
fuel_type
diesel      594
gasoline    594
Name: count, dtype: int64


,country_code,fuel_type,year,tax_value,tax_eur_litre,unit,product
10,AUT,diesel,2005,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
11,AUT,diesel,2006,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
12,AUT,diesel,2007,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
13,AUT,diesel,2008,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
14,AUT,diesel,2009,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
15,AUT,diesel,2010,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
16,AUT,diesel,2011,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
17,AUT,diesel,2012,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
18,AUT,diesel,2013,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...
19,AUT,diesel,2014,20.0,0.02,1000 l,Gas oil automobile Automotive gas oil Dieselkr...


In [93]:
#Países sin impuestos en tax_yearly

tax_missing = tax_yearly[tax_yearly["tax_eur_litre"].isna()].copy()

print("Filas sin impuestos:", len(tax_missing))

print("\nPaíses sin impuestos:")
print(sorted(tax_missing["country_code"].dropna().unique()))

print("\nDetalle por país y combustible:")
print(
    tax_missing
    .groupby(["country_code", "fuel_type"])
    .size()
    .reset_index(name="filas_sin_impuesto")
)

Filas sin impuestos: 0

Países sin impuestos:
[]

Detalle por país y combustible:
Empty DataFrame
Columns: [country_code, fuel_type, filas_sin_impuesto]
Index: []


In [94]:
##Países disponibles en tax_fuel

print("Países en tax_fuel:")
print(sorted(tax_fuel["country_code"].dropna().unique()))

print("\nPaíses en prices_clean:")
print(sorted(prices_clean["country_code"].dropna().unique()))

missing_tax_countries = sorted(
    set(prices_clean["country_code"].dropna().unique()) -
    set(tax_fuel["country_code"].dropna().unique())
)

print("\nPaíses que están en precios pero no en impuestos:")
print(missing_tax_countries)

Países en tax_fuel:
['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']

Países en prices_clean:
['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']

Países que están en precios pero no en impuestos:
[]


**Creamos el dataset unificado**

In [96]:
entrada_union = len(prices_clean)

dataset = prices_clean.copy()

# Unión con PIB per cápita
dataset = dataset.merge(
    gdp_ready[["country_code", "country_name", "year", "gdp_per_capita"]],
    on=["country_code", "year"],
    how="left"
)

# Unión con dependencia energética
dataset = dataset.merge(
    energy_ready[["country_code", "year", "energy_dependency"]],
    on=["country_code", "year"],
    how="left"
)

# Unión con impuestos
dataset = dataset.merge(
    tax_yearly[["country_code", "year", "fuel_type", "tax_value", "tax_eur_litre"]],
    on=["country_code", "year", "fuel_type"],
    how="left"
)

dataset = dataset.sort_values(["country_code", "fuel_type", "date"]).copy()

add_tracking(
    fase="union_fuentes",
    dataset="dataset_unificado",
    registros_entrada=entrada_union,
    registros_salida=len(dataset),
    eliminados=0,
    motivo="Unión de precios con PIB per cápita, dependencia energética e impuestos por país, año y tipo de combustible"
)

dataset.head(20)

,date,year,country_code,country_code_2,fuel_type,price_original,price_eur_litre,unit,raw_product,country_name,gdp_per_capita,energy_dependency,tax_value,tax_eur_litre
2123,2005-01-03,2005,AUT,AT,diesel,859.0,0.859,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2122,2005-01-10,2005,AUT,AT,diesel,851.0,0.851,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2121,2005-01-17,2005,AUT,AT,diesel,836.0,0.836,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2120,2005-01-24,2005,AUT,AT,diesel,840.0,0.840,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2119,2005-01-31,2005,AUT,AT,diesel,855.0,0.855,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2118,2005-02-07,2005,AUT,AT,diesel,854.0,0.854,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2117,2005-02-14,2005,AUT,AT,diesel,840.0,0.840,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2116,2005-02-21,2005,AUT,AT,diesel,850.0,0.850,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2115,2005-02-28,2005,AUT,AT,diesel,853.0,0.853,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2114,2005-03-07,2005,AUT,AT,diesel,865.0,0.865,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02


In [97]:
# Comprobación del dataset unificado

print("Filas dataset:", len(dataset))
print("Columnas dataset:", len(dataset.columns))

print("\nNulos por columna:")
print(dataset.isna().sum())

print("\nPorcentaje de nulos:")
print((dataset.isna().mean() * 100).round(2))

print("\nTipos de combustible:")
print(dataset["fuel_type"].value_counts())

print("\nPaíses:")
print(dataset["country_code"].nunique())

dataset.head(20)

Filas dataset: 56011
Columnas dataset: 14

Nulos por columna:
date                     0
year                     0
country_code             0
country_code_2           0
fuel_type                0
price_original           0
price_eur_litre          0
unit                     0
raw_product              0
country_name           918
gdp_per_capita        3726
energy_dependency    28853
tax_value                0
tax_eur_litre            0
dtype: int64

Porcentaje de nulos:
date                  0.00
year                  0.00
country_code          0.00
country_code_2        0.00
fuel_type             0.00
price_original        0.00
price_eur_litre       0.00
unit                  0.00
raw_product           0.00
country_name          1.64
gdp_per_capita        6.65
energy_dependency    51.51
tax_value             0.00
tax_eur_litre         0.00
dtype: float64

Tipos de combustible:
fuel_type
diesel      28006
gasoline    28005
Name: count, dtype: int64

Países:
27


,date,year,country_code,country_code_2,fuel_type,price_original,price_eur_litre,unit,raw_product,country_name,gdp_per_capita,energy_dependency,tax_value,tax_eur_litre
2123,2005-01-03,2005,AUT,AT,diesel,859.0,0.859,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2122,2005-01-10,2005,AUT,AT,diesel,851.0,0.851,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2121,2005-01-17,2005,AUT,AT,diesel,836.0,0.836,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2120,2005-01-24,2005,AUT,AT,diesel,840.0,0.840,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2119,2005-01-31,2005,AUT,AT,diesel,855.0,0.855,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2118,2005-02-07,2005,AUT,AT,diesel,854.0,0.854,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2117,2005-02-14,2005,AUT,AT,diesel,840.0,0.840,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2116,2005-02-21,2005,AUT,AT,diesel,850.0,0.850,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2115,2005-02-28,2005,AUT,AT,diesel,853.0,0.853,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02
2114,2005-03-07,2005,AUT,AT,diesel,865.0,0.865,1000 l,AT_price_with_tax_diesel__Gas oil automobile A...,Austria,38157.370229,NaN,20.0,0.02


In [98]:
# Tracking de control de calidad final

nulos_finales = dataset.isna().sum().to_dict()

add_tracking(
    fase="control_calidad_final",
    dataset="dataset_unificado",
    registros_entrada=len(dataset),
    registros_salida=len(dataset),
    eliminados=0,
    motivo=(
        "Revisión final de calidad. Se mantienen algunos nulos porque no todas las fuentes "
        "tienen cobertura completa para todos los países o años. "
        f"Nulos finales por columna: {nulos_finales}. "
        "La dependencia energética empieza más tarde que la serie de precios. "
        "Los impuestos se han expandido por país, año y combustible usando forward fill y backward fill."
    )
)

In [99]:
#Dataset final

dataset_final = dataset[[
    "date",
    "year",
    "country_code",
    "country_code_2",
    "country_name",
    "fuel_type",
    "price_original",
    "price_eur_litre",
    "gdp_per_capita",
    "energy_dependency",
    "tax_value",
    "tax_eur_litre"
]].copy()

dataset_final = dataset_final.drop_duplicates()

print("Filas dataset_final:", len(dataset_final))
print("Columnas dataset_final:", len(dataset_final.columns))

dataset_final.head(20)

Filas dataset_final: 56011
Columnas dataset_final: 12


,date,year,country_code,country_code_2,country_name,fuel_type,price_original,price_eur_litre,gdp_per_capita,energy_dependency,tax_value,tax_eur_litre
2123,2005-01-03,2005,AUT,AT,Austria,diesel,859.0,0.859,38157.370229,NaN,20.0,0.02
2122,2005-01-10,2005,AUT,AT,Austria,diesel,851.0,0.851,38157.370229,NaN,20.0,0.02
2121,2005-01-17,2005,AUT,AT,Austria,diesel,836.0,0.836,38157.370229,NaN,20.0,0.02
2120,2005-01-24,2005,AUT,AT,Austria,diesel,840.0,0.840,38157.370229,NaN,20.0,0.02
2119,2005-01-31,2005,AUT,AT,Austria,diesel,855.0,0.855,38157.370229,NaN,20.0,0.02
2118,2005-02-07,2005,AUT,AT,Austria,diesel,854.0,0.854,38157.370229,NaN,20.0,0.02
2117,2005-02-14,2005,AUT,AT,Austria,diesel,840.0,0.840,38157.370229,NaN,20.0,0.02
2116,2005-02-21,2005,AUT,AT,Austria,diesel,850.0,0.850,38157.370229,NaN,20.0,0.02
2115,2005-02-28,2005,AUT,AT,Austria,diesel,853.0,0.853,38157.370229,NaN,20.0,0.02
2114,2005-03-07,2005,AUT,AT,Austria,diesel,865.0,0.865,38157.370229,NaN,20.0,0.02


In [100]:
# Guardar dataset unificado

output_dataset = PROCESSED_DIR / "dataset_unificado.csv"

dataset_final.to_csv(
    output_dataset,
    index=False,
    encoding="utf-8"
)

print("Dataset unificado guardado en:", output_dataset)
print("Filas:", len(dataset_final))
print("Columnas:", len(dataset_final.columns))

Dataset unificado guardado en: data/processed/dataset_unificado.csv
Filas: 56011
Columnas: 12


In [106]:
# Limpiar duplicados del tracking

fases_limpias = []
vistos = set()

for fase in pipeline_tracking["fases"]:
    clave = (
        fase["fase"],
        fase["dataset"],
        fase["registros_entrada"],
        fase["registros_salida"],
        fase["registros_eliminados"],
        fase["motivo"]
    )

    if clave not in vistos:
        fases_limpias.append(fase)
        vistos.add(clave)

pipeline_tracking["fases"] = fases_limpias

print("Número de fases finales en tracking:", len(pipeline_tracking["fases"]))

Número de fases finales en tracking: 7


In [107]:
# Guardar pipeline_tracking.json

output_tracking = PROCESSED_DIR / "pipeline_tracking.json"

with open(output_tracking, "w", encoding="utf-8") as f:
    json.dump(pipeline_tracking, f, indent=4, ensure_ascii=False)

print("Tracking guardado en:", output_tracking)

Tracking guardado en: data/processed/pipeline_tracking.json


In [102]:
!ls -lah data/processed

total 4.7M
drwxr-xr-x 2 root root 4.0K May 12 10:44 .
drwxr-xr-x 4 root root 4.0K May 12 08:58 ..
-rw-r--r-- 1 root root 4.7M May 12 10:44 dataset_unificado.csv
-rw-r--r-- 1 root root 4.8K May 12 09:45 impuestos_procesados.csv
-rw-r--r-- 1 root root 5.1K May 12 10:44 pipeline_tracking.json


In [104]:
##Comprobar que el CSV final se puede leer
dataset_check = pd.read_csv("data/processed/dataset_unificado.csv")

print(dataset_check.shape)
dataset_check.head()

(56011, 12)


,date,year,country_code,country_code_2,country_name,fuel_type,price_original,price_eur_litre,gdp_per_capita,energy_dependency,tax_value,tax_eur_litre
0,2005-01-03,2005,AUT,AT,Austria,diesel,859.0,0.859,38157.370229,NaN,20.0,0.02
1,2005-01-10,2005,AUT,AT,Austria,diesel,851.0,0.851,38157.370229,NaN,20.0,0.02
2,2005-01-17,2005,AUT,AT,Austria,diesel,836.0,0.836,38157.370229,NaN,20.0,0.02
3,2005-01-24,2005,AUT,AT,Austria,diesel,840.0,0.840,38157.370229,NaN,20.0,0.02
4,2005-01-31,2005,AUT,AT,Austria,diesel,855.0,0.855,38157.370229,NaN,20.0,0.02


In [108]:
##Comprobar el tracking
with open("data/processed/pipeline_tracking.json", "r", encoding="utf-8") as f:
    tracking_check = json.load(f)

tracking_check

{'fecha_ejecucion': '2026-05-12 09:01:38',
 'fases': [{'fase': 'extraccion_transformacion',
   'dataset': 'PIB_per_capita',
   'registros_entrada': 266,
   'registros_salida': 17556,
   'registros_eliminados': 0,
   'motivo': 'Conversión de formato ancho a largo y eliminación de filas sin país o año'},
  {'fase': 'limpieza',
   'dataset': 'dependencia_energetica',
   'registros_entrada': 409,
   'registros_salida': 409,
   'registros_eliminados': 0,
   'motivo': 'Eliminación de filas vacías, duplicados y limpieza de blancos'},
  {'fase': 'limpieza',
   'dataset': 'impuestos_oil_bulletin',
   'registros_entrada': 35,
   'registros_salida': 31,
   'registros_eliminados': 4,
   'motivo': 'Limpieza de filas vacías, duplicados, blancos y nombres de columnas'},
  {'fase': 'limpieza_transformacion',
   'dataset': 'impuestos_oil_bulletin',
   'registros_entrada': 192,
   'registros_salida': 132,
   'registros_eliminados': 60,
   'motivo': 'Reconstrucción de cabecera, transformación a formato l

In [110]:
!ls -lah data/processed

total 4.7M
drwxr-xr-x 2 root root 4.0K May 12 10:44 .
drwxr-xr-x 4 root root 4.0K May 12 08:58 ..
-rw-r--r-- 1 root root 4.7M May 12 10:44 dataset_unificado.csv
-rw-r--r-- 1 root root 4.8K May 12 09:45 impuestos_procesados.csv
-rw-r--r-- 1 root root 3.1K May 12 10:53 pipeline_tracking.json
